# Convert-Pheno CLI tutorial

This notebook runs the latest Convert-Pheno source from GitHub and uses
the small regression fixtures included in the repository. It demonstrates:

- Beacon v2 Models Format (BFF) `individuals` to Phenopackets v2 (PXF)
- PXF to BFF `individuals` and `biosamples`
- OMOP-CDM CSV tables to BFF
- REDCap to BFF with a mapping file and terminology audit
- Raw CSV to BFF with the same mapping model

The commands are ordinary `convert-pheno` CLI commands. The same commands
can be used in a terminal after replacing the example paths with your own.


## 1. Install the current source

A Colab runtime is temporary. The following cell starts from a clean clone
of the latest `main` branch. The notebook prints the exact Git commit later
so that a run can still be identified and reproduced.


In [ ]:
%cd /content
!rm -rf convert-pheno
!git clone --depth 1 https://github.com/CNAG-Biomedical-Informatics/convert-pheno.git
%cd /content/convert-pheno


Install the Linux and Perl dependencies. CPAN installation can take several
minutes in a new runtime.


In [ ]:
!apt-get update -qq
!DEBIAN_FRONTEND=noninteractive apt-get install -y -qq cpanminus libbz2-dev zlib1g-dev libperl-dev libssl-dev
!cpanm --notest --installdeps .


Record both the software version and source revision. These values are more
useful for reproducibility than a version hard-coded into the notebook.


In [ ]:
!bin/convert-pheno --version
!git rev-parse HEAD


Run the active regression suite before using the tutorial fixtures. This
verifies the installation and the same routes demonstrated below.


In [ ]:
!prove -lr t


## 2. Prepare an output directory

All generated files will be kept under `/content/convert-pheno-output`.
They can be inspected or downloaded from Colab's Files panel.


In [ ]:
import json
from pathlib import Path

output_dir = Path("/content/convert-pheno-output")
output_dir.mkdir(exist_ok=True)

def read_json(path):
    with Path(path).open(encoding="utf-8") as handle:
        return json.load(handle)

print(output_dir)


## 3. Convert BFF to PXF

The first route converts a Beacon `individuals` collection into a JSON
array of Phenopackets. The input is the BFF fixture used by the regression
suite.


In [ ]:
!bin/convert-pheno \
  -ibff t/bff2pxf/in/individuals.json \
  -opxf /content/convert-pheno-output/phenopackets.json \
  -O


In [ ]:
phenopackets = read_json(output_dir / "phenopackets.json")
first = phenopackets[0]

print("Phenopackets:", len(phenopackets))
print("First subject:", first["subject"]["id"])
print("Measurements for first subject:", len(first.get("measurements", [])))


## 4. Convert PXF to multiple BFF entities

When `--entities` is used with `-obff`, Convert-Pheno writes one file per
requested Beacon entity. This example extracts both `individuals` and
`biosamples` from a Phenopackets collection.


In [ ]:
!mkdir -p /content/convert-pheno-output/pxf-to-bff
!bin/convert-pheno \
  -ipxf t/pxf2bff/in/pxf.json \
  -obff \
  --entities individuals biosamples \
  --out-dir /content/convert-pheno-output/pxf-to-bff \
  -O


In [ ]:
entity_dir = output_dir / "pxf-to-bff"
individuals = read_json(entity_dir / "individuals.json")
biosamples = read_json(entity_dir / "biosamples.json")

print("Individuals:", len(individuals))
print("Biosamples:", len(biosamples))
print("First biosample:", biosamples[0]["id"])


## 5. Convert OMOP-CDM CSV tables to BFF

`-iomop` accepts multiple OMOP table files. This compact example provides
`PERSON`, `CONCEPT`, and `DRUG_EXPOSURE`. Original OMOP columns are retained
under BFF `info` by default for provenance and cross-checking; use
`--no-source-info` only when those source values should be omitted.


In [ ]:
!bin/convert-pheno \
  -iomop \
  t/omop2bff/in/PERSON.csv \
  t/omop2bff/in/CONCEPT.csv \
  t/omop2bff/in/DRUG_EXPOSURE.csv \
  -obff /content/convert-pheno-output/omop-individuals.json \
  -O


In [ ]:
omop_individuals = read_json(output_dir / "omop-individuals.json")
first = omop_individuals[0]

print("Individuals:", len(omop_individuals))
print("First individual:", first["id"])
print("Provenance sections:", sorted(first.get("info", {}).keys()))


## 6. Convert REDCap to BFF and audit terminology searches

REDCap conversion requires three project-specific inputs:

1. A REDCap data export
2. Its REDCap data dictionary
3. A Convert-Pheno mapping file

`--term-audit` records how source terms were resolved. The output format is selected from `.tsv`, `.tsv.gz`, or `.xlsx`. The default
search mode is `exact`; review the audit before accepting mapped terms.


In [ ]:
!bin/convert-pheno \
  -iredcap t/redcap2bff/in/redcap_data.csv \
  --redcap-dictionary t/redcap2bff/in/redcap_dictionary.csv \
  --mapping-file t/redcap2bff/in/redcap_mapping.yaml \
  -obff /content/convert-pheno-output/redcap-individuals.json \
  --term-audit /content/convert-pheno-output/redcap-terminology.tsv \
  -O


In [ ]:
redcap_individuals = read_json(output_dir / "redcap-individuals.json")
print("Individuals:", len(redcap_individuals))
print("First individual:", redcap_individuals[0]["id"])


In [ ]:
!head -n 6 /content/convert-pheno-output/redcap-terminology.tsv


## 7. Convert raw CSV to BFF

Raw CSV conversion needs the data file and a Convert-Pheno mapping file,
but not a REDCap data dictionary. Set `--sep` explicitly when the input
delimiter differs from the default.


In [ ]:
!bin/convert-pheno \
  -icsv t/csv2bff/in/csv_data.csv \
  --mapping-file t/csv2bff/in/csv_mapping.yaml \
  --sep , \
  -obff /content/convert-pheno-output/csv-individuals.json \
  -O


In [ ]:
csv_individuals = read_json(output_dir / "csv-individuals.json")
first = csv_individuals[0]

print("Individuals:", len(csv_individuals))
print("First individual:", first["id"])
print("Provenance sections:", sorted(first.get("info", {}).keys()))


## 8. Use your own files

Upload files through Colab's Files panel and replace the fixture paths in
the relevant command. Files in `/content` disappear when the runtime is
recycled, so download any results you need to retain.

Convert-Pheno does not send conversion inputs to a Convert-Pheno service.
Google Colab is nevertheless a third-party cloud environment. Do not upload
identifiable or otherwise sensitive clinical data unless that use is
permitted by your institution and data-governance requirements. For
controlled data, run the same CLI locally or in an approved container
environment.


## Next steps

- [CLI reference](https://cnag-biomedical-informatics.github.io/convert-pheno/use-as-a-command-line-interface/)
- [Choose a conversion](https://cnag-biomedical-informatics.github.io/convert-pheno/conversion-recipes/)
- [Mapping-file guide](https://cnag-biomedical-informatics.github.io/convert-pheno/mapping-files/)
- [Development validation](https://cnag-biomedical-informatics.github.io/convert-pheno/development-validation/)

The repository's [`t/` directory](https://github.com/CNAG-Biomedical-Informatics/convert-pheno/tree/main/t)
contains additional inputs and expected outputs for every maintained route.
